<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1_DPP%26DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Component based data preprocessing

Task performed by the component

Crime Hotspot prediction
The model predicts the probability or density of crime occurrences across a geographic grid(based on GM) for a future time window -> (OUTPUT – risk score or heatmap)

Crime Type Classification

Given a set of conditions (Time, Location, Weather, Land Use), the model predicts the most likely type of crime to occur. -> (OUTPUT – crime type)

Spatio-Temporal Forecasting

forecast future counts of crimes in specific regions(GMs) -> (OUTPUT – crime count for a specific cell and time bine)

##Preprocessing & Data Cleaning steps performed


Remove Irrelevant Columns

Handling Typos/Inconsistencies

Coordinate Validation

Datetime Conversion

Boolean Mapping

Categorical Encoding

Temporal Decomposition



##Component Based Preprocessing

In [42]:
import pandas as pd
import numpy as np

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
crime_df = pd.read_csv('/content/drive/MyDrive/DSGP/CrimeData_v3.csv')

In [44]:
crime_df

,crime,location,date,sex,victim_ethnicity,victim_age,time,weather,latitude,longitude,holiday_name,is_holiday,land_use_type,gn_division,gn_pcode,gn_population,gn_distance_m,victim_ethnicity,status_report
0,burglary,mulgampola,2019-12-31,f,muslim,54,08:17:00,Cloudy,7.280544,80.616500,Non-Holiday,0,General Urban,Welata,LK2130170,21826,311.8,muslim,Valid
1,burglary,car park,2020-01-04,m,sinhala,42,02:00:00,Rainy,7.283445,80.619385,Non-Holiday,0,Commercial,Katukele West,LK2130105,8913,352.4,sinhala,Valid
2,theft,bus stand,2020-01-08,f,sinhala,20,21:01:00,Rainy,7.256425,80.590461,Eid al-Adha,1,Commercial,Penideniya,LK2139135,16411,518.6,sinhala,Valid
3,vehicle theft,aniwatte,2020-01-10,m,sinhala,29,12:10:00,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,1,General Urban,Aniwatta East,LK2130100,1107,381.5,sinhala,Valid
4,robbery,dutugamunu mawatha,2020-01-11,m,sinhala,59,02:39:00,Rainy,7.312344,80.645687,Non-Holiday,0,General Urban,Aruppala East,LK2130050,1293,322.6,sinhala,Valid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,drugs,walespark,2025-07-30,m,sinhala,65,09:19:00,Rainy,7.291772,80.636707,Non-Holiday,0,Commercial,Mahanuwara,LK2130120,11613,518.0,sinhala,Valid
1604,drugs,peradeniya road,2025-08-16,m,sinhala,59,04:38:00,Rainy,7.280930,80.619477,Non-Holiday,0,General Urban,Suduhumpala West,LK2130165,18808,353.7,sinhala,Valid
1605,robbery,provincial education department,2025-09-13,f,sinhala,50,07:25:00,Rainy,7.290341,80.633563,Non-Holiday,0,Commercial,Bogambara,LK2130145,2198,315.0,sinhala,Valid
1606,robbery,post office,2025-09-17,f,sinhala,61,20:12:00,Rainy,7.292186,80.633475,Non-Holiday,0,Commercial,Ihala Katukele,LK2130115,6925,335.7,sinhala,Valid


### Spatial Feature Aggregation

This code converts your list of individual crime incidents into a neat summary table that shows the total number of crimes for each neighborhood on each day.

In [45]:
#The primary key -> gn_pcode
# Group by Date and GN Division
aggregated_df = crime_df.groupby(['date', 'gn_pcode', 'gn_division', 'gn_population','land_use_type']).size().reset_index(name='crime_count')
# Sort by date and division for a clear timeline
aggregated_df = aggregated_df.sort_values(by=['date', 'gn_division'])


In [33]:
# Display the result
#print("First 10 rows of Aggregated Data:")
#print(aggregated_df.head(10))

### Time-Binning

Give every crime in your original list a label (Morning, Night, etc.).

In [46]:
#Create time bins-> devide the 24-hour day in to blocks

#Extract hour and create bins
crime_df['hour'] = pd.to_datetime(crime_df['time']).dt.hour

# Define bins 0-6 (Night), 6-12 (Morning), 12-18 (Afternoon), 18-24 (Evening)
bins = [0, 6, 12, 18, 24]
labels = ['Night', 'Morning', 'Afternoon', 'Evening']
crime_df['time_bin'] = pd.cut(crime_df['hour'], bins=bins, labels=labels, right=False)


/tmp/ipython-input-1384983575.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df['hour'] = pd.to_datetime(crime_df['time']).dt.hour


### Grid Resampling

 Create the "Empty Calendar" that includes every neighborhood and every time slot

In [47]:
# Create a master grid
# 1. Get all unique dates, time bins, and GN PCodes
all_dates = pd.date_range(start=crime_df['date'].min(), end=crime_df['date'].max(), freq='D')
all_bins = labels
all_gn = crime_df[['gn_pcode', 'gn_division', 'gn_population', 'land_use_type']].drop_duplicates()

# Use MultiIndex to create every possible combination
index = pd.MultiIndex.from_product([all_dates, all_bins, all_gn['gn_pcode']],
                                   names=['date', 'time_bin', 'gn_pcode'])
master_grid = pd.DataFrame(index=index).reset_index()

# Merge back the GN info (population/land use) to the master grid
master_grid = master_grid.merge(all_gn, on='gn_pcode', how='left')


### Negative Sampling

Fill the empty boxes in your calendar with "0" so the model knows no crime happened there


In [48]:
# Aggregate actual crime into tye grid
# Group actual data by date, bin, and GN
actual_counts = crime_df.groupby(['date', 'time_bin', 'gn_pcode']).size().reset_index(name='crime_count')

# Ensure types match for merging
actual_counts['date'] = pd.to_datetime(actual_counts['date'])
master_grid['date'] = pd.to_datetime(master_grid['date'])

#Merge and Fill Zeros
# Merge Master Grid with Actual Counts
final_grid = pd.merge(master_grid, actual_counts, on=['date', 'time_bin', 'gn_pcode'], how='left')

# This is the 'Negative Sampling' part
final_grid['crime_count'] = final_grid['crime_count'].fillna(0).astype(int)

/tmp/ipython-input-2154198557.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  actual_counts = crime_df.groupby(['date', 'time_bin', 'gn_pcode']).size().reset_index(name='crime_count')


In [49]:
print(final_grid.head(50))

         date time_bin   gn_pcode             gn_division  gn_population  \
0  2019-12-31    Night  LK2130170                  Welata          21826   
1  2019-12-31    Night  LK2130170                  Welata          21826   
2  2019-12-31    Night  LK2130105           Katukele West           8913   
3  2019-12-31    Night  LK2130105           Katukele West           8913   
4  2019-12-31    Night  LK2139135              Penideniya          16411   
5  2019-12-31    Night  LK2130100           Aniwatta East           1107   
6  2019-12-31    Night  LK2130050           Aruppala East           1293   
7  2019-12-31    Night  LK2130050           Aruppala East           1293   
8  2019-12-31    Night  LK2130165        Suduhumpala West          18808   
9  2019-12-31    Night  LK2130165        Suduhumpala West          18808   
10 2019-12-31    Night  LK2130065               Mahaiyawa          11543   
11 2019-12-31    Night  LK2130065               Mahaiyawa          11543   
12 2019-12-3

### Trend and Seasonality Extraction




In [50]:
#1. Extract Basic Time Components

# Extract basic features from the 'date' column
final_grid['month'] = final_grid['date'].dt.month
final_grid['day_of_week'] = final_grid['date'].dt.dayofweek  # 0=Monday, 6=Sunday
final_grid['is_weekend'] = final_grid['date'].dt.dayofweek.isin([5, 6]).astype(int)

#2. Map the Season - > weather patterns
def get_season(month):
    if month in [3, 4, 5]: return 'First_Inter_Monsoon'
    elif month in [6, 7, 8, 9]: return 'South_West_Monsoon'
    elif month in [10, 11]: return 'Second_Inter_Monsoon'
    else: return 'North_East_Monsoon'

final_grid['season'] = final_grid['month'].apply(get_season)

#3. Map Holiday
# Create a lookup from your original crime_df
holiday_lookup = crime_df[['date', 'is_holiday', 'holiday_name']].drop_duplicates('date')

# Merge it into your final_grid
holiday_lookup['date'] = pd.to_datetime(holiday_lookup['date'])
final_grid['date'] = pd.to_datetime(final_grid['date'])

final_grid = final_grid.merge(holiday_lookup, on='date', how='left')

# Fill the gaps (the 0-crime days usually aren't holidays)
final_grid['is_holiday'] = final_grid['is_holiday'].fillna(0)
final_grid['holiday_name'] = final_grid['holiday_name'].fillna('None')

#4. Trend Indicator
# Create a simple day counter from the start of the dataset
start_date = final_grid['date'].min()
final_grid['days_from_start'] = (final_grid['date'] - start_date).dt.days


### Lagged Feature Engineering

In [51]:
# 1. Sort the Data Correctly
# Ensure the grid is perfectly ordered by place and time
final_grid = final_grid.sort_values(by=['gn_pcode', 'date', 'time_bin'])

#2. Create the "Previous Slot" Lag (Short0-term Memory)
final_grid['lag_1_slot'] = final_grid.groupby('gn_pcode')['crime_count'].shift(1)

#3. Create "Same Time Yesterday" Lag()Daily Seasonality
final_grid['lag_daily'] = final_grid.groupby('gn_pcode')['crime_count'].shift(4)

#4. Create a "Moving Average" (Trend Memory)
# Rolling average of the last 4 slots
final_grid['rolling_avg_24h'] = final_grid.groupby('gn_pcode')['crime_count'].transform(lambda x: x.shift(1).rolling(window=4).mean())

#5. Handle the "Empty Memory"
# Fill the first few days of empty memory with 0
final_grid[['lag_1_slot', 'lag_daily', 'rolling_avg_24h']] = final_grid[['lag_1_slot', 'lag_daily', 'rolling_avg_24h']].fillna(0)


### Spatial Lag Features

In [52]:
# 1. Create a 'District Proxy' from gn_pcode
final_grid['district_proxy'] = final_grid['gn_pcode'].str[:4]

# 2. Calculate the average crime in that district for EACH time slot
# We save this as a new column first so we can shift it properly
final_grid['district_avg_count'] = final_grid.groupby(['date', 'time_bin', 'district_proxy'])['crime_count'].transform('mean')

# 3. Create the Spatial Lag (Corrected Shift)
# We group by gn_pcode and shift the 'district_avg_count' column
final_grid['spatial_lag_1_slot'] = final_grid.groupby('gn_pcode')['district_avg_count'].shift(1)

# 4. Cleanup: Fill the very first rows with 0 and drop the temporary proxy columns
final_grid['spatial_lag_1_slot'] = final_grid['spatial_lag_1_slot'].fillna(0)
final_grid = final_grid.drop(columns=['district_proxy', 'district_avg_count'])


### Population Normalization

In [53]:
#1. Calculate the Crime Rate
# Calculate Crime Rate per 1,000 people
# We add a small constant (1) to the denominator if any population is 0 to avoid errors
final_grid['crime_rate_per_1000'] = (final_grid['crime_count'] / final_grid['gn_population']) * 1000

#2. Normalize the Features -> population and crime count
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
final_grid[['pop_scaled', 'count_scaled']] = scaler.fit_transform(final_grid[['gn_population', 'crime_count']])

#3. Handle the Log Transformation
final_grid['crime_count_log'] = np.log1p(final_grid['crime_count'])


### Data Integrity & NaN Scrubbing

In [54]:
# List all columns that depend on past data
lag_cols = ['lag_1_slot', 'lag_daily', 'rolling_avg_24h', 'spatial_lag_1_slot']

# Drop rows where we don't have historical data
print(f"Rows before scrubbing: {len(final_grid)}")
final_grid = final_grid.dropna(subset=lag_cols)
print(f"Rows after scrubbing: {len(final_grid)}")

# Final Sanity Check for NaNs
if final_grid[lag_cols].isnull().sum().sum() == 0:
    print("✅ All NaNs removed. Dataset is mathematically stable.")

Rows before scrubbing: 1566312
Rows after scrubbing: 1566312
✅ All NaNs removed. Dataset is mathematically stable.


### Encoding Categorical Variables

In [55]:
# 1. One-Hot Encode
final_grid = pd.get_dummies(final_grid, columns=['land_use_type', 'season'], prefix=['land', 'season'])

### Cyclical Temporal Transformation

In [57]:
#1. Identify the cyclic  features
# -> month, day_of_week, time_bin

#2.  Map bins to numbers if they are strings (e.g., 'Morning' -> 0)
bin_mapping = {'Morning': 0, 'Afternoon': 1, 'Evening': 2, 'Night': 3}
final_grid['time_bin_num'] = final_grid['time_bin'].map(bin_mapping)

#3. Apply Sine and Cosine Transformations
# 1. Month (Cycle of 12)
final_grid['month_sin'] = np.sin(2 * np.pi * final_grid['month'] / 12)
final_grid['month_cos'] = np.cos(2 * np.pi * final_grid['month'] / 12)

# 2. Day of Week (Cycle of 7)
final_grid['day_sin'] = np.sin(2 * np.pi * final_grid['day_of_week'] / 7)
final_grid['day_cos'] = np.cos(2 * np.pi * final_grid['day_of_week'] / 7)

# 3. Time Bin (Cycle of 4)
final_grid['time_sin'] = np.sin(2 * np.pi * final_grid['time_bin_num'] / 4)
final_grid['time_cos'] = np.cos(2 * np.pi * final_grid['time_bin_num'] / 4)

#4. Drop the Original Linear Columns
final_grid = final_grid.drop(columns=['month', 'day_of_week', 'time_bin', 'time_bin_num'])

### Temporal Train-Test Split

In [58]:
# Sort Chronologically
final_grid = final_grid.sort_values(by='days_from_start').reset_index(drop=True)

# Calculate split point
split_idx = int(len(final_grid) * 0.8)

# Split the WHOLE dataframe
train_df = final_grid.iloc[:split_idx].copy()
test_df = final_grid.iloc[split_idx:].copy()

### Target Encoding

In [59]:
# Calculate mean risk per GN using ONLY training data
pcode_risk_map = train_df.groupby('gn_pcode')['crime_count'].mean()
global_mean = train_df['crime_count'].mean()

# Map the risk to both sets
train_df['gn_pcode_encoded'] = train_df['gn_pcode'].map(pcode_risk_map)
test_df['gn_pcode_encoded'] = test_df['gn_pcode'].map(pcode_risk_map).fillna(global_mean)

### Feature Selection

- victim_age, victim_sex, victim_ethnicity, offense_category.

In [61]:
#1. Separate the "Potential Targets"
# Create a dictionary or for different goals (3)

#2. Remove Administrative & Redundant Info
cols_to_drop = [
    'date', 'gn_division', 'holiday_name','gn_pcode',
    'crime_count', 'crime_rate_per_1000',
    'count_scaled', 'crime_count_log'
]

# Create Features (X)
X_train = train_df.drop(columns=cols_to_drop)
X_test = test_df.drop(columns=cols_to_drop)

# Create Targets (y) for your 3 goals
y_train_forecast = train_df['crime_count']
y_test_forecast = test_df['crime_count']

y_train_hotspot = (train_df['crime_count'] > 0).astype(int)
y_test_hotspot = (test_df['crime_count'] > 0).astype(int)


In [65]:
X_train.columns

Index(['gn_population', 'is_weekend', 'is_holiday', 'days_from_start',
       'lag_1_slot', 'lag_daily', 'rolling_avg_24h', 'spatial_lag_1_slot',
       'pop_scaled', 'land_Commercial', 'land_General Urban',
       'land_Recreational', 'land_Residential', 'season_First_Inter_Monsoon',
       'season_North_East_Monsoon', 'season_Second_Inter_Monsoon',
       'season_South_West_Monsoon', 'month_sin', 'month_cos', 'day_sin',
       'day_cos', 'time_sin', 'time_cos', 'gn_pcode_encoded'],
      dtype='object')

In [67]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1253049 entries, 0 to 1253048
Data columns (total 24 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   gn_population                1253049 non-null  int64  
 1   is_weekend                   1253049 non-null  int64  
 2   is_holiday                   1253049 non-null  float64
 3   days_from_start              1253049 non-null  int64  
 4   lag_1_slot                   1253049 non-null  float64
 5   lag_daily                    1253049 non-null  float64
 6   rolling_avg_24h              1253049 non-null  float64
 7   spatial_lag_1_slot           1253049 non-null  float64
 8   pop_scaled                   1253049 non-null  float64
 9   land_Commercial              1253049 non-null  bool   
 10  land_General Urban           1253049 non-null  bool   
 11  land_Recreational            1253049 non-null  bool   
 12  land_Residential             1253049 non-n

### Feature Scaling

In [68]:
from sklearn.preprocessing import StandardScaler

# Define the columns that need scaling, the continuous numerical ones
cols_to_scale = [
    'gn_population',
    'days_from_start',
    'lag_1_slot',
    'lag_daily',
    'rolling_avg_24h',
    'spatial_lag_1_slot',
    'pop_scaled',
    'gn_pcode_encoded'

]

scaler = StandardScaler()

scaler.fit(X_train[cols_to_scale])

# 2. Transform the Training data
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])

# 3. Transform the Test data using the TRAINING parameters
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


Save the updated dataset to the google

In [69]:
# Save the feature sets
X_train.to_csv('/content/drive/MyDrive/DSGP/X_train_final.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/DSGP/X_test_final.csv', index=False)

# Save the target sets (e.g., for the hotspot goal)
y_train_hotspot.to_csv('/content/drive/MyDrive/DSGP/y_train_hotspot.csv', index=False)
y_test_hotspot.to_csv('/content/drive/MyDrive/DSGP/y_test_hotspot.csv', index=False)

print("Files successfully saved to your Google Drive!")

Files successfully saved to your Google Drive!
